## Аналіз A/B-тестів

Ви - аналітик даних в ІТ-компанії і до вас надійшла задача проаналізувати дані A/B тесту в популярній [грі Cookie Cats](https://www.facebook.com/cookiecatsgame). Це - гра-головоломка в стилі «з’єднай три», де гравець повинен з’єднати плитки одного кольору, щоб очистити дошку та виграти рівень. На дошці також зображені співаючі котики :)

Під час проходження гри гравці стикаються з воротами, які змушують їх чекати деякий час, перш ніж вони зможуть прогресувати або зробити покупку в додатку.

У цьому блоці завдань ми проаналізуємо результати A/B тесту, коли перші ворота в Cookie Cats було переміщено з рівня 30 на рівень 40. Зокрема, ми хочемо зрозуміти, як це вплинуло на утримання (retention) гравців. Тобто хочемо зрозуміти, чи переміщення воріт на 10 рівнів пізніше якимось чином вплинуло на те, що користувачі перестають грати в гру раніше чи пізніше з точки зору кількості їх днів з моменту встановлення гри.

Будемо працювати з даними з файлу `cookie_cats.csv`. Колонки в даних наступні:

- `userid` - унікальний номер, який ідентифікує кожного гравця.
- `version` - чи потрапив гравець в контрольну групу (gate_30 - ворота на 30 рівні) чи тестову групу (gate_40 - ворота на 40 рівні).
- `sum_gamerounds` - кількість ігрових раундів, зіграних гравцем протягом першого тижня після встановлення
- `retention_1` - чи через 1 день після встановлення гравець повернувся і почав грати?
- `retention_7` - чи через 7 днів після встановлення гравець повернувся і почав грати?

Коли гравець встановлював гру, його випадковим чином призначали до групи gate_30 або gate_40.

1. Для початку, уявімо, що ми тільки плануємо проведення зазначеного А/B-тесту і хочемо зрозуміти, дані про скількох користувачів нам треба зібрати, аби досягнути відчутного ефекту. Відчутним ефектом ми вважатимемо збільшення утримання на 1% після внесення зміни. Обчисліть, скільки користувачів сумарно нам треба аби досягнути такого ефекту, якщо продакт менеджер нам повідомив, що базове утримання є 19%.

In [22]:
import numpy as np
import pandas as pd
import scipy.stats as stats
import statsmodels.stats.api as sms
import matplotlib as mpl
import matplotlib.pyplot as plt
import seaborn as sns
from math import ceil
from statsmodels.stats.proportion import proportions_ztest, proportion_confint

%matplotlib inline

In [7]:
effect_size = sms.proportion_effectsize(0.19, 0.20)# Розрахунок розміру ефекту на основі наших очікуваних показників
effect_size 

np.float64(-0.025241594409087353)

In [8]:
required_n = sms.NormalIndPower().solve_power(
    effect_size,
    power=0.8,
    alpha=0.05,
    ratio=1
    )                                                  # Розрахунок необхідного розміру вибірки

required_n = ceil(required_n)                          # Округлення до наступного цілого числа

print(required_n)

24638


Нам буде потрібно мінімум 24638 спостережень для кожної групи.


2. Зчитайте дані АВ тесту у змінну `df` та виведіть середнє значення показника показник `retention_7` (утримання на 7 день) по версіям гри. Сформулюйте гіпотезу: яка версія дає краще утримання через 7 днів після встановлення гри?

In [6]:
df = pd.read_csv('../data/cookie_cats.csv')

df.head()

,userid,version,sum_gamerounds,retention_1,retention_7
0,116,gate_30,3,False,False
1,337,gate_30,38,True,False
2,377,gate_40,165,True,False
3,483,gate_40,1,False,False
4,488,gate_40,179,True,True


In [10]:
retention = df.groupby('version')['retention_7'].mean()
print(retention)

version
gate_30    0.190201
gate_40    0.182000
Name: retention_7, dtype: float64


H₀: retention_7 (gate_30) = retention_7 (gate_40)
Hₐ: retention_7 (gate_30) > retention_7 (gate_40)

версія gate_30 дає краще утримання

3. Перевірте з допомогою пасуючого варіанту z-тесту, чи дає якась з версій гри кращий показник `retention_7` на рівні значущості 0.05. Обчисліть також довірчі інтервали для варіантів до переміщення воріт і після. Виведіть результат у форматі:

    ```
    z statistic: ...
    p-value: ...
    Довірчий інтервал 95% для групи control: [..., ...]
    Довірчий інтервал 95% для групи treatment: [..., ...]
    ```

    де замість `...` - обчислені значення.
    
    В якості висновку дайте відповідь на два питання:  

      1. Чи є статистична значущою різниця між поведінкою користувачів у різних версіях гри?   
      2. Чи перетинаються довірчі інтервали утримання користувачів з різних версій гри? Про що це каже?  


In [26]:

# групи
control = df[df['version'] == 'gate_30']
treatment = df[df['version'] == 'gate_40']

# кількість успіхів (retention_7 = 1)
count = [control['retention_7'].sum(), treatment['retention_7'].sum()]

# розмір вибірок
nobs = [len(control), len(treatment)]
successes = [
    control['retention_7'].sum(),
    treatment['retention_7'].sum()
]

In [27]:
successes

[np.int64(8502), np.int64(8279)]

In [29]:
z_stat, pval = proportions_ztest(successes, nobs=nobs)

lower_con, upper_con = proportion_confint(successes[0], nobs[0], alpha=0.05)
lower_treat, upper_treat = proportion_confint(successes[1], nobs[1], alpha=0.05)


print(f'z statistic: {z_stat:.2f}')
print(f'p-value: {pval:.3f}')
print(f'Довірчий інтервал 95% для групи control: [{lower_con:.3f}, {upper_con:.3f}]')
print(f'Довірчий інтервал 95% для групи treatment: [{lower_treat:.3f}, {upper_treat:.3f}]')

z statistic: 3.16
p-value: 0.002
Довірчий інтервал 95% для групи control: [0.187, 0.194]
Довірчий інтервал 95% для групи treatment: [0.178, 0.186]


A p-value = 0.002 < 0.05
ми відхиляємо H₀
 це означає:

Є статистично значущі докази того, що поведінка користувачів (retention_7) відрізняється між версіями гри.

B
Довірчі інтервали не перетинаються, що свідчить про наявність статистично значущої різниці між групами та підтверджує, що версія з воротами на рівні 30 має вище утримання користувачів.

4. Виконайте тест Хі-квадрат на рівні значущості 5% аби визначити, чи є залежність між версією гри та утриманням гравця на 7ий день після реєстрації.

    - Напишіть, як для цього тесту будуть сформульовані гіпотези.
    - Проведіть обчислення, виведіть p-значення і напишіть висновок за результатами тесту.


H₀: version і retention_7 незалежні  
Hₐ: version і retention_7 залежні

In [32]:
from scipy.stats import chi2_contingency


contingency_table = pd.crosstab(df['version'], df['retention_7'])

chi2, p_value, dof, expected = chi2_contingency(contingency_table)

print("Contingency table:")
print(contingency_table)

print(f"\nChi2 statistic: {chi2:.3f}")
print(f"p-value: {p_value:.5f}")

Contingency table:
retention_7  False  True 
version                  
gate_30      36198   8502
gate_40      37210   8279

Chi2 statistic: 9.959
p-value: 0.00160


p-value =0.00160 < 0.05

відхиляємо H₀
Є статистично значущі докази того, що версія гри та утримання користувача на 7-й день є залежними.